# Analisis de Cavidades en RuBisCO — CASTpFold

Pipeline: CASTpFold → APBS → Analisis

Basado en Poudel et al. (2020) y CASTpFold (Ye et al. 2024, NAR)

**CASTpFold** reemplaza a fpocket. Usa el servidor web sts.bioe.uic.edu/castpdev/

Instrucciones: ejecutar en orden. CASTpFold tarda ~10-30 seg por PDB.

## 0. Imports y configuracion

In [ ]:
import sys
sys.path.insert(0, '/content/analisismolecular/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import urllib.request
import tempfile
import time

from libreria_analisismolecular import castpfold as cpf

sns.set_style('whitegrid')
print('Imports OK')
%matplotlib inline

## 1. Descargar PDBs de RuBisCO

In [ ]:
# PDBs de RuBisCO por grupo (Poudel 2020)
PDBS = {
    '4RUB':  {'group': 'G-I',   'S': 77},
    '1GK8':  {'group': 'G-I',   'S': 61},
    '1RBL':  {'group': 'G-II',  'S': 13},
    '1GEH':  {'group': 'G-II',  'S': 15},
    '2RUS':  {'group': 'G-III', 'S': 5},
}

PDB_DIR = Path('/content/pdb')
PDB_DIR.mkdir(exist_ok=True)

for pid in PDBS:
    fp = PDB_DIR / f'{pid}.pdb'
    if not fp.exists():
        url = f'https://files.rcsb.org/download/{pid}.pdb'
        urllib.request.urlretrieve(url, fp)
        print(f'Descargado {pid}')

print(f'Total: {len(list(PDB_DIR.glob("*.pdb")))} PDBs')

## 2. Extraer cadena A (subunidad grande con sitio activo)

In [ ]:
from Bio.PDB import PDBParser, PDBIO, Select

class ChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id

CHAIN_A_DIR = Path('/content/pdb/chain_a')
CHAIN_A_DIR.mkdir(exist_ok=True)

parser = PDBParser(QUIET=True)
io = PDBIO()

for pid in PDBS:
    pdb_path = PDB_DIR / f'{pid}.pdb'
    out_path = CHAIN_A_DIR / f'{pid}_A.pdb'
    
    if out_path.exists():
        continue
    
    try:
        structure = parser.get_structure(pid, str(pdb_path))
        io.set_structure(structure)
        io.save(str(out_path), ChainSelect('A'))
        print(f'Extraida cadena A de {pid}')
    except Exception as e:
        print(f'Error con {pid}: {e}')

print(f'Cadenas A listas: {len(list(CHAIN_A_DIR.glob("*.pdb")))}')

## 3. Correr CASTpFold sobre cada PDB

In [ ]:
# CASTpFold usa el servidor web — agregar delays para no sobrecargar
RESULTS_DIR = Path('/content/castpfold_results')
RESULTS_DIR.mkdir(exist_ok=True)

all_cavities = []

for pdb_file in sorted(CHAIN_A_DIR.glob('*.pdb')):
    pid = pdb_file.stem.replace('_A', '')
    info = PDBS.get(pid, {})
    
    print(f'\nProcesando {pid} (grupo={info.get("group", "?")}, S={info.get("S", "?")})...')
    
    try:
        output_dir = RESULTS_DIR / pid
        
        # CASTpFold: enviar PDB al servidor y descargar resultados
        results_dir = cpf.run_castpfold(
            str(pdb_file),
            output_dir=str(output_dir),
            probe_radius=1.4,
            compute_pocket=True,
            wait_time=20,
            extra_wait=30,
            retries=3,
        )
        
        # Parsear resultados
        cavities = cpf.parse_castpfold_results(results_dir)
        
        if not cavities.empty:
            cavities['pdb'] = pid
            cavities['group'] = info.get('group', 'unknown')
            cavities['S'] = info.get('S', np.nan)
            all_cavities.append(cavities)
            print(f'  -> {len(cavities)} cavidades detectadas')
        else:
            print(f'  -> No se detectaron cavidades')
        
        # Delay entre PDBs para no sobrecargar el servidor
        time.sleep(5)
        
    except Exception as e:
        print(f'  -> Error: {e}')

if all_cavities:
    df_all = pd.concat(all_cavities, ignore_index=True)
    print(f'\nTotal cavidades: {len(df_all)}')
    display(df_all.head())
else:
    df_all = pd.DataFrame()
    print('\nNo se encontraron cavidades.')

## 4. Filtrar cavidades por volumen

In [ ]:
if not df_all.empty:
    # Filtrar por volumen minimo (Angstroms cubicos)
    df_filtered = cpf.filter_cavities_by_volume(df_all, volume_threshold=50.0)
    print(f'Cavidades antes del filtro: {len(df_all)}')
    print(f'Cavidades despues del filtro (vol >= 50 A^3): {len(df_filtered)}')
    
    # Estadisticas por grupo
    if 'volume' in df_filtered.columns:
        display(df_filtered.groupby('group')['volume'].agg(['mean', 'std', 'count']).round(2))

## 5. Correlacion cavidades vs especificidad (S)

In [ ]:
if not df_all.empty and 'volume' in df_all.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Volumen vs S
    sns.scatterplot(
        data=df_all, x='volume', y='S',
        hue='group', style='group', s=120, ax=axes[0]
    )
    axes[0].set_xlabel('Volumen de cavidad (A^3)')
    axes[0].set_ylabel('Especificidad CO2/O2 (S)')
    axes[0].set_title('Volumen vs Especificidad')
    
    # Area vs S (si existe columna area)
    if 'area' in df_all.columns:
        sns.scatterplot(
            data=df_all, x='area', y='S',
            hue='group', style='group', s=120, ax=axes[1]
        )
        axes[1].set_xlabel('Area de cavidad (A^2)')
        axes[1].set_title('Area vs Especificidad')
    
    plt.tight_layout()
    plt.show()
    
    # Correlacion
    if 'volume' in df_all.columns:
        corr_vol = df_all['volume'].corr(df_all['S'])
        print(f'Correlacion volumen-S: {corr_vol:.3f}')
    if 'area' in df_all.columns:
        corr_area = df_all['area'].corr(df_all['S'])
        print(f'Correlacion area-S: {corr_area:.3f}')
else:
    print('No hay datos suficientes para correlacion.')

## 6. Electrostatica con APBS (opcional)

In [ ]:
# APBS para calcular carga electrostatica de cavidades
# Descomentar cuando APBS este instalado (via condacolab)

# from libreria_analisismolecular import cavidades
# 
# for pdb_file in sorted(CHAIN_A_DIR.glob('*.pdb')):
#     pid = pdb_file.stem.replace('_A', '')
#     print(f'\nAPBS para {pid}...')
#     try:
#         apbs_dir = cavidades.run_apbs(str(pdb_file))
#         print(f'  -> APBS completo: {apbs_dir}')
#     except Exception as e:
#         print(f'  -> Error: {e}')

## 7. Resumen y conclusiones

In [ ]:
if not df_all.empty:
    print('=== Resumen de cavidades por CASTpFold ===')
    print(f'Total PDBs analizados: {df_all["pdb"].nunique()}')
    print(f'Total cavidades detectadas: {len(df_all)}')
    print()
    
    # Por grupo
    cols = [c for c in ['volume', 'area', 'depth'] if c in df_all.columns]
    if cols:
        display(df_all.groupby('group')[cols + ['S']].agg(['mean', 'count']).round(2))
    
    print('\n=== Observaciones ===')
    print('- CASTpFold detecta bolsillos y cavidades superficiales')
    print('- G-I (4RUB, 1GK8): mayor S, cavidades mas grandes?')
    print('- G-II (1RBL, 1GEH): bolsas aisladas, menor S?')
    print('- G-III (2RUS): valor atipico, S~5')
    print('\nProximo paso: correlacion con campos electricos (Nivel 3)')